# Multi-Class Classification with Hidden Layers

## The Softmax Function

$$
o = z W_o + b_o, \qquad o_j = \sum_{m=1}^{k} z_m w_{o,mj} + b_{o,j}, \quad j = 1, \ldots, K
$$

$$
\hat{y}_j = \text{softmax}(o)_j = \frac{e^{o_j}}{\sum_{m=1}^K e^{o_m}}, \qquad j = 1, \ldots, K
$$

### Why *soft*max?

## Softmax Reduces to the Sigmoid for two classes

## The Network in Matrix Notation

### The model in matrix notation

$$
\begin{align*}
H & = X W_h + b_h \\
Z & = \varphi(H) \\
O & = Z W_o + b_o \\
\hat{Y} & = \text{softmax}(O) \\
L & = -\frac{1}{n} \sum_{i=1}^n \sum_{j=1}^K y_{ij} \log \hat{y}_{ij}
\end{align*}
$$

### Gradients

$$
\begin{align*}
\frac{\partial L}{\partial W_o} & = -\frac{1}{n} Z^\top E &
\frac{\partial L}{\partial b_o} & = -\frac{1}{n} \mathbf{1}^\top E \\
\frac{\partial L}{\partial W_h} & = -\frac{1}{n} X^\top \Delta &
\frac{\partial L}{\partial b_h} & = -\frac{1}{n} \mathbf{1}^\top \Delta
\end{align*}
$$

## Implementation

### Softmax, forward pass and gradients in NumPy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [ ]:
def softmax(O, axis=-1):
    # Subtract the maximum for numerical stability (the result does not change)
    shifted = O - np.max(O, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)

logits = np.array([2.0, 1.0, -3.0])

hard_max = (logits == np.max(logits)).astype(float)
softmax_probs = softmax(logits)

print("Logits:", logits)
print("Hard max:", hard_max)
print("Exponentiated Logits:", np.exp(logits))
print("Softmax Probabilities:", softmax_probs)
print("Sum of Softmax Probabilities:", np.sum(softmax_probs))

# Shift invariance: the same probabilities, even for huge logits where np.exp would overflow
print("Softmax of shifted logits:", softmax(logits + 1000))

In [ ]:
def relu(H):
    return np.maximum(0, H)

def relu_prime(H):
    return (H > 0).astype(float)

def forward(X, params, activation=relu):
    W_h, b_h, W_o, b_o = params
    H = X @ W_h + b_h
    Z = activation(H)
    O = Z @ W_o + b_o
    return H, Z, softmax(O)

def cross_entropy(Y, Y_hat):
    return -np.mean(np.sum(Y * np.log(Y_hat), axis=1))

def gradients(X, Y, params, activation=relu, activation_prime=relu_prime):
    W_h, b_h, W_o, b_o = params
    n = X.shape[0]

    H, Z, Y_hat = forward(X, params, activation)
    E = Y - Y_hat

    dW_o = -(Z.T @ E) / n
    db_o = -E.sum(axis=0, keepdims=True) / n

    Delta = (E @ W_o.T) * activation_prime(H)
    dW_h = -(X.T @ Delta) / n
    db_h = -Delta.sum(axis=0, keepdims=True) / n

    return dW_h, db_h, dW_o, db_o

In [ ]:
def make_spirals(n_per_class=100, n_classes=3, noise=0.2, seed=0):
    rng = np.random.default_rng(seed)
    X = np.zeros((n_per_class * n_classes, 2))
    y = np.zeros(n_per_class * n_classes, dtype=int)
    for c in range(n_classes):
        idx = slice(n_per_class * c, n_per_class * (c + 1))
        radius = np.linspace(0.05, 1, n_per_class)
        angle = np.linspace(4 * c, 4 * (c + 1), n_per_class) + rng.normal(0, noise, n_per_class)
        X[idx] = np.c_[radius * np.sin(angle), radius * np.cos(angle)]
        y[idx] = c
    return X, y

X_spiral, y_spiral = make_spirals()
Y_spiral = np.eye(3)[y_spiral]          # one-hot matrix Y (n x K)

plt.figure(figsize=(6, 6))
plt.scatter(X_spiral[:, 0], X_spiral[:, 1], c=y_spiral, cmap='viridis', edgecolor='k', s=40)
plt.title('Three spirals')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def init_params(n_features, n_hidden, n_classes, seed=0):
    rng = np.random.default_rng(seed)
    return [
        rng.normal(0, 0.5, (n_features, n_hidden)),   # W_h
        np.zeros((1, n_hidden)),                      # b_h
        rng.normal(0, 0.5, (n_hidden, n_classes)),    # W_o
        np.zeros((1, n_classes)),                     # b_o
    ]

def tanh_prime(H):
    return 1 - np.tanh(H) ** 2

activations = {
    "relu": (relu, relu_prime, torch.relu),
    "tanh": (np.tanh, tanh_prime, torch.tanh),
}

params_spiral = init_params(n_features=2, n_hidden=16, n_classes=3)

### Training with the derived gradients

In [ ]:
def train(X, Y, params, activation, activation_prime, learning_rate=0.5, epochs=5000):
    params = [p.copy() for p in params]
    losses = []
    for epoch in range(epochs):
        grads = gradients(X, Y, params, activation, activation_prime)
        for param, grad in zip(params, grads):
            param -= learning_rate * grad            # theta <- theta - eta * dL/dtheta
        losses.append(cross_entropy(Y, forward(X, params, activation)[2]))
    return params, losses

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

xx_s, yy_s = np.meshgrid(np.linspace(-1.2, 1.2, 200), np.linspace(-1.2, 1.2, 200))
grid_s = np.c_[xx_s.ravel(), yy_s.ravel()]

for ax, (name, (act, act_prime, _)) in zip(axes[1:], activations.items()):
    trained, losses = train(X_spiral, Y_spiral, params_spiral, act, act_prime)
    axes[0].plot(losses, label=name)

    accuracy = (forward(X_spiral, trained, act)[2].argmax(axis=1) == y_spiral).mean()
    print(f"{name}: final loss {losses[-1]:.4f}, training accuracy {accuracy:.3f}")

    predicted_class = forward(grid_s, trained, act)[2].argmax(axis=1).reshape(xx_s.shape)
    ax.contourf(xx_s, yy_s, predicted_class, levels=[-0.5, 0.5, 1.5, 2.5], cmap='viridis', alpha=0.4)
    ax.scatter(X_spiral[:, 0], X_spiral[:, 1], c=y_spiral, cmap='viridis', edgecolor='k', s=25)
    ax.set_title(f'Decision regions ({name})')

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Training loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### The same network in PyTorch

In [ ]:
trained_np, losses_np = train(X_spiral, Y_spiral, params_spiral, relu, relu_prime)

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 3)).double()

with torch.no_grad():
    W_h0, b_h0, W_o0, b_o0 = params_spiral
    model[0].weight.copy_(torch.tensor(W_h0.T))
    model[0].bias.copy_(torch.tensor(b_h0.ravel()))
    model[2].weight.copy_(torch.tensor(W_o0.T))
    model[2].bias.copy_(torch.tensor(b_o0.ravel()))

criterion = nn.CrossEntropyLoss()                     # averages over the observations
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

X_tensor = torch.tensor(X_spiral)
labels_tensor = torch.tensor(y_spiral)

for epoch in range(5000):
    loss = criterion(model(X_tensor), labels_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"Final loss: PyTorch {loss.item():.6f}, NumPy {losses_np[-1]:.6f}")
print("Largest difference in W_h:", np.abs(model[0].weight.detach().numpy().T - trained_np[0]).max())
print("Largest difference in W_o:", np.abs(model[2].weight.detach().numpy().T - trained_np[2]).max())

## Application: the Bigram Model as a Softmax Network

In [ ]:
alice_text = """
Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do. Once or twice she had peeped into the book her
sister was reading, but it had no pictures or conversations in it.
"""

import re
tokens = re.findall(r"[a-z']+", alice_text.lower())

vocab = sorted(set(tokens))
vocab_size = len(vocab)
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

print(f"Vocabulary size: {vocab_size}")

# Bigrams: (current word, next word)
bigrams = list(zip(tokens[:-1], tokens[1:]))
print(f"Number of bigrams: {len(bigrams)}")

# Input: one-hot row vectors, target: index of the next word
X_bigram = torch.eye(vocab_size)[[word_to_idx[current] for current, _ in bigrams]]
y_bigram = torch.tensor([word_to_idx[following] for _, following in bigrams])

print("X_bigram shape:", X_bigram.shape, " y_bigram shape:", y_bigram.shape)

In [ ]:
counts = np.zeros((vocab_size, vocab_size))
for current, following in bigrams:
    counts[word_to_idx[current], word_to_idx[following]] += 1

row_totals = counts.sum(axis=1, keepdims=True)
count_probs = np.divide(counts, row_totals, out=np.zeros_like(counts), where=row_totals > 0)

# Average entropy of the bigram table over the observed bigrams = the smallest possible cross-entropy loss
minimum_loss = -np.mean([np.log(count_probs[word_to_idx[current], word_to_idx[following]])
                         for current, following in bigrams])
print(f"Smallest possible cross-entropy loss: {minimum_loss:.4f}")

In [ ]:
def fit(model, epochs, learning_rate=0.05):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    for epoch in range(epochs):
        loss = criterion(model(X_bigram), y_bigram)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return loss.item()

# Without a hidden layer: V x V logits (the model returns logits, the softmax is inside the loss)
torch.manual_seed(0)
bigram_model = nn.Linear(vocab_size, vocab_size)
print(f"No hidden layer:   loss {fit(bigram_model, epochs=1000):.4f}")

# With a (linear) hidden layer of width d: the weights of the first layer are the word embeddings
embedding_size = 8
torch.manual_seed(0)
bigram_model_hidden = nn.Sequential(
    nn.Linear(vocab_size, embedding_size, bias=False),
    nn.Linear(embedding_size, vocab_size),
)
print(f"Hidden layer d={embedding_size}: loss {fit(bigram_model_hidden, epochs=2000):.4f}")

n_parameters = lambda model: sum(p.numel() for p in model.parameters())
print("Parameters:", n_parameters(bigram_model), "vs", n_parameters(bigram_model_hidden))

In [ ]:
def next_word_probs(model, word):
    with torch.no_grad():
        logits = model(torch.eye(vocab_size)[word_to_idx[word]])
        return torch.softmax(logits, dim=0).numpy()

for word in ["alice", "of", "sister"]:
    top = np.argsort(-count_probs[word_to_idx[word]])[:3]
    print(f"After '{word}':")
    for j in top:
        print(f"  {idx_to_word[j]:10s} counting {count_probs[word_to_idx[word], j]:.2f}"
              f"   no hidden layer {next_word_probs(bigram_model, word)[j]:.2f}"
              f"   hidden layer {next_word_probs(bigram_model_hidden, word)[j]:.2f}")

In [ ]:
# Generate text by always choosing the most likely next word
def generate_sentence(start_word, model, length=8):
    sentence = [start_word]
    for _ in range(length - 1):
        probs = next_word_probs(model, sentence[-1])
        sentence.append(idx_to_word[int(np.argmax(probs))])
    return " ".join(sentence)

print(generate_sentence("alice", bigram_model))